# Exercise 04 — Add a Potassium A-Current to the HH Model

**Module 04 | Estimated time: 2–3 hours**

---

## Background

The **A-type potassium current** ($I_A$) is a rapidly inactivating outward current present in many real neurons. It delays the first spike after a long hyperpolarisation — a phenomenon called **first-spike latency**. This is computationally important for temporal coding in the auditory and sensory systems.

The A-current is modelled as:

$$I_A = g_A \cdot a^3 \cdot b \cdot (V - E_K)$$

With gating variables $a$ (activation) and $b$ (inactivation):

$$\alpha_a(V) = \frac{0.02(V + 65)}{1 - e^{-(V+65)/10}}, \quad \beta_a(V) = 0.01(V + 65)e^{-(V+65)/10}$$

$$\alpha_b(V) = 0.003e^{-(V+65)/30}, \quad \beta_b(V) = \frac{0.015}{1 + e^{-(V+35)/10}}$$

The voltage equation becomes:

$$C_m \frac{dV}{dt} = -g_{Na}m^3h(V-E_{Na}) - g_Kn^4(V-E_K) - g_L(V-E_L) - I_A + I_{ext}$$

## Tasks

1. Add `alpha_a`, `beta_a`, `alpha_b`, `beta_b` device functions
2. Add state arrays `a[N]` and `b[N]` (SoA)
3. Update `hh_deriv` to include $I_A$
4. Add $da/dt$ and $db/dt$ to the Euler step
5. Compare f-I curves with and without $I_A$
6. *(Challenge)* Measure first-spike latency as a function of I

In [ ]:
!nvidia-smi

In [ ]:
%%writefile hh_a_current.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_Cm, c_gNa, c_gK, c_gL, c_gA;
__constant__ float c_ENa, c_EK, c_EL, c_dt;

// Standard HH rate functions (copy from lecture)
__device__ __forceinline__ float alpha_m(float V) {
    float d=V+40.f; return fabsf(d)<1e-5f?1.f:0.1f*d/(1.f-expf(-d/10.f)); }
__device__ __forceinline__ float beta_m(float V)  { return 4.f*expf(-(V+65.f)/18.f); }
__device__ __forceinline__ float alpha_h(float V) { return 0.07f*expf(-(V+65.f)/20.f); }
__device__ __forceinline__ float beta_h(float V)  { return 1.f/(1.f+expf(-(V+35.f)/10.f)); }
__device__ __forceinline__ float alpha_n(float V) {
    float d=V+55.f; return fabsf(d)<1e-5f?0.1f:0.01f*d/(1.f-expf(-d/10.f)); }
__device__ __forceinline__ float beta_n(float V)  { return 0.125f*expf(-(V+65.f)/80.f); }

// TODO 1: Add A-current rate functions
__device__ __forceinline__ float alpha_a(float V) {
    // TODO: implement alpha_a
    return ???;
}
__device__ __forceinline__ float beta_a(float V) {
    // TODO: implement beta_a
    return ???;
}
__device__ __forceinline__ float alpha_b(float V) {
    // TODO: implement alpha_b
    return ???;
}
__device__ __forceinline__ float beta_b(float V) {
    // TODO: implement beta_b
    return ???;
}

// TODO 2: Update hh_deriv to include 6 state variables (V, m, h, n, a, b)
// and return 6 derivatives (dV, dm, dh, dn, da, db)
__device__ __forceinline__ void hh_deriv_with_A(
    float V, float m, float h, float n, float a, float b, float I,
    float* dV, float* dm, float* dh, float* dn, float* da, float* db
) {
    float INa = c_gNa*m*m*m*h*(V-c_ENa);
    float IK  = c_gK*n*n*n*n*(V-c_EK);
    float IL  = c_gL*(V-c_EL);
    // TODO: add I_A term
    float IA  = ???;
    *dV = (I - INa - IK - IL - IA) / c_Cm;
    *dm = alpha_m(V)*(1.f-m) - beta_m(V)*m;
    *dh = alpha_h(V)*(1.f-h) - beta_h(V)*h;
    *dn = alpha_n(V)*(1.f-n) - beta_n(V)*n;
    // TODO: add da, db
    *da = ???;
    *db = ???;
}

// TODO 3: Write the kernel that simulates N neurons and counts spikes
// (for the f-I sweep, similar to hh_sweep_kernel from Lecture 3)
__global__ void hh_a_sweep(
    const float* I_arr, int* spike_count,
    int N, int T_steps, int skip_steps
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    float V=-65.f, I=I_arr[i];
    // TODO: initialise m, h, n at steady state
    float m=???, h=???, n=???;
    // TODO: initialise a, b at steady state
    float a=???, b=???;

    int spikes=0, above=0;
    for(int step=0;step<T_steps;step++) {
        float dV,dm,dh,dn,da,db;
        hh_deriv_with_A(V,m,h,n,a,b,I,&dV,&dm,&dh,&dn,&da,&db);
        V+=c_dt*dV;
        m=fmaxf(0.f,fminf(1.f,m+c_dt*dm));
        h=fmaxf(0.f,fminf(1.f,h+c_dt*dh));
        n=fmaxf(0.f,fminf(1.f,n+c_dt*dn));
        // TODO: update a and b with bounds check
        ???;
        if(step>=skip_steps) {
            if(V>0.f&&!above){spikes++;above=1;}
            if(V<-30.f) above=0;
        }
    }
    spike_count[i]=spikes;
}

int main() {
    const int N=2000;
    const float T_ms=500.f, T_skip=100.f, dt=0.01f;
    const float gA=4.0f;  // A-current max conductance

    float Cm=1.f,gNa=120.f,gK=36.f,gL=0.3f;
    float ENa=50.f,EK=-77.f,EL=-54.4f;
    CUDA_CHECK(cudaMemcpyToSymbol(c_Cm,&Cm,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_gNa,&gNa,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_gK,&gK,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_gL,&gL,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_gA,&gA,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_ENa,&ENa,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_EK,&EK,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_EL,&EL,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,&dt,4));

    float *hI=(float*)malloc(N*4);
    for(int i=0;i<N;i++) hI[i]=30.f*i/(N-1.f);

    float *dI; int *dsc;
    CUDA_CHECK(cudaMalloc(&dI,N*4)); CUDA_CHECK(cudaMalloc(&dsc,N*4));
    CUDA_CHECK(cudaMemcpy(dI,hI,N*4,cudaMemcpyHostToDevice));

    int thr=256,blk=(N+thr-1)/thr;
    int T_steps=(int)(T_ms/dt), skip=(int)(T_skip/dt);

    hh_a_sweep<<<blk,thr>>>(dI,dsc,N,T_steps,skip);
    CUDA_CHECK(cudaDeviceSynchronize());

    int *hsc=(int*)malloc(N*4);
    CUDA_CHECK(cudaMemcpy(hsc,dsc,N*4,cudaMemcpyDeviceToHost));

    FILE* f=fopen("fi_with_A.txt","w");
    float Teff=(T_ms-T_skip)/1000.f;
    for(int i=0;i<N;i++) fprintf(f,"%.4f %.2f\n",hI[i],hsc[i]/Teff);
    fclose(f);
    printf("Done. f-I curve written to fi_with_A.txt\n");

    free(hI); free(hsc); cudaFree(dI); cudaFree(dsc);
    return 0;
}

In [ ]:
!nvcc -O2 -o hh_a_current hh_a_current.cu -lm && ./hh_a_current

In [ ]:
# Compare f-I curves with and without I_A
import numpy as np
import matplotlib.pyplot as plt

# Load both curves
try:
    d_std = np.loadtxt('fi_curve_hh.txt')   # from Lecture 3
    d_a   = np.loadtxt('fi_with_A.txt')

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(d_std[:,0], d_std[:,1], 'b-', lw=2, label='Standard HH (no I_A)')
    ax.plot(d_a[:,0],   d_a[:,1],   'r-', lw=2, label='HH + A-current')
    ax.set_xlabel('Input current (μA/cm²)', fontsize=13)
    ax.set_ylabel('Firing rate (Hz)', fontsize=13)
    ax.set_title('Effect of A-Current on f-I Curve', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("Observation: I_A shifts the rheobase and reduces firing rates at low I.")
except:
    print("Generate fi_curve_hh.txt from Lecture 3 first.")

---
Check against [ex04_solution.ipynb](ex04_solution.ipynb) when done.